## Setup

Import required modules and configure settings.

# LoRA Fine-Tuning Pipeline

This notebook runs the three phases of the fine-tuning pipeline:
1. **Generate synthetic data** (compliant/adversarial/edge-case examples)
2. **Train LoRA adapter** on Vertex AI
3. **Evaluate model** (F1 score ≥ 0.95 to pass)

**Prerequisites:** `gcloud auth application-default login`

In [ ]:
import sys
import os
from src.tuning.tuning_generator import TuningGenerator

sys.path.insert(0, '/path/to/agentics-sdlc')  # noqa: E402
os.chdir('/path/to/agentics-sdlc')  # noqa: E402

print("✓ Setup complete - ready for Phase 1")

## Phase 1: Generate Synthetic Data

Extract policies from `.agent/` directory and generate training + validation datasets.

In [ ]:
import asyncio

async def phase1_generate_data():
    """Generate synthetic training and validation datasets."""
    
    gen = TuningGenerator()
    train_uri, val_uri = await gen.tuning_generator_generate_datasets()
    print(f"Training data:   {train_uri}")
    print(f"Validation data: {val_uri}")
    return train_uri, val_uri

# Execute Phase 1
train_uri, val_uri = asyncio.run(phase1_generate_data())

## Phase 2: Train LoRA Adapter

Submit Vertex AI SFT job and deploy the tuned endpoint.

In [ ]:
from src.tuning.tuning_train import TuningTrain

async def phase2_train_adapter():
    """Train LoRA adapter on Vertex AI and deploy endpoint."""
    
    trainer = TuningTrain()
    endpoint_id = await trainer.tuning_train_orchestrate_pipeline(train_uri, val_uri)
    print(f"Training complete. Endpoint ID: {endpoint_id}")
    return endpoint_id

# Execute Phase 2 (training typically takes 30-60 minutes)
endpoint_id = asyncio.run(phase2_train_adapter())

## Phase 3: Evaluate Model

Validate the tuned model against the validation dataset.

In [ ]:
from src.tuning.tuning_evaluate import tuning_evaluate_tuned_endpoint

async def phase3_evaluate_model():
    """Evaluate tuned model against validation dataset."""
    
    metrics = await tuning_evaluate_tuned_endpoint(
        endpoint_id=endpoint_id,
        validation_uri=val_uri
    )
    print(f"Precision: {metrics['precision']:.3f}")
    print(f"Recall:    {metrics['recall']:.3f}")
    print(f"F1:        {metrics['f1']:.3f}")
    print(f"Status:    {'PASSED' if metrics['passes_threshold'] else 'REVIEW NEEDED'}")
    return metrics

# Execute Phase 3
metrics = asyncio.run(phase3_evaluate_model())

## Summary

Three-phase LoRA fine-tuning pipeline:

| Phase | Duration | Output |
|-------|----------|--------|
| **Phase 1** | 5-10 min | Training + validation datasets (gs://bucket/*.jsonl) |
| **Phase 2** | 30-60 min | Tuned endpoint (projects/.../endpoints/...) |
| **Phase 3** | 5-10 min | Metrics (F1 ≥ 0.95 to pass) |

**Configuration:** Edit hyperparameters in `src/tuning/tuning_utils.py` before Phase 2.